# 44 — FIM-Modulated Learning: Does Self-Observation Accelerate Adaptation?

**UNPRECEDENTED:** sc-neurocore asked whether FIM should interact with STDP.
We advised "additive, separate." But what if FIM MODULATES the learning rate?

Hypothesis: when the system is near sync (high R, high FIM sensitivity),
learning should be FASTER because the system "knows" it's close to the
target state. When desynchronised (low R), learning should be slower
because the system doesn't have a clear signal.

This is FIM-modulated plasticity: η_eff = η₀ × h(R)
where h(R) = 1/(1-R²+ε) is the FIM sensitivity.

## Tests
1. Can FIM-modulated coupling learn the correct K_nm from random IC?
2. Does FIM-modulated learning converge faster than standard?
3. Is the learned coupling more or less stable?

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27

N = 8
K_target = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]


def fim_sensitivity(phases, eps=0.01, cap=50.0):
    R = float(np.abs(np.mean(np.exp(1j * phases))))
    return min(1.0 / (1.0 - R**2 + eps), cap)


def hebbian_update(theta, K, eta):
    """Hebbian/STDP-like coupling update: ΔK_ij ∝ cos(θ_j - θ_i).
    Pairs that are in-phase get strengthened."""
    diff = theta[None, :] - theta[:, None]
    dK = eta * np.cos(diff)
    np.fill_diagonal(dK, 0)
    return dK


def simulate_learning(
    N,
    K_target,
    omega,
    mode="standard",
    eta=0.001,
    fim_lambda=1.0,
    dt=0.02,
    T=200.0,
    noise=0.05,
    seed=42,
):
    """Simulate Kuramoto + Hebbian learning with optional FIM modulation.

    Modes:
    - 'standard': fixed learning rate η
    - 'fim_modulated': η_eff = η × h(R)
    - 'fim_gated': η_eff = η × h(R), PLUS FIM phase correction
    """
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    K_learned = rng.uniform(0, 0.1, (N, N))
    K_learned = (K_learned + K_learned.T) / 2
    np.fill_diagonal(K_learned, 0)

    n_steps = int(T / dt)
    K_SCALE = 5.0  # operating coupling strength

    correlation_history = []
    R_history = []

    for s in range(n_steps):
        K_eff = K_learned * K_SCALE
        diff = theta[None, :] - theta[:, None]
        coupling = np.sum(K_eff * np.sin(diff), axis=1) / N
        dphi = omega + coupling

        # FIM phase correction (always active in fim modes)
        if mode in ("fim_modulated", "fim_gated"):
            z = np.exp(1j * theta)
            mu = np.angle(np.mean(z))
            R = np.abs(np.mean(z))
            phase_diff = (mu - theta + np.pi) % (2 * np.pi) - np.pi
            sensitivity = min(1.0 / (1.0 - R**2 + 0.01), 50.0)
            dphi += fim_lambda * (1.0 / N) * np.sin(phase_diff) * sensitivity

        theta = (theta + dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)) % (2 * np.pi)

        # Learning step
        if mode == "standard":
            eta_eff = eta
        elif mode in ("fim_modulated", "fim_gated"):
            eta_eff = eta * fim_sensitivity(theta)

        dK = hebbian_update(theta, K_learned, eta_eff)
        K_learned += dt * dK
        K_learned = (K_learned + K_learned.T) / 2  # enforce symmetry
        np.fill_diagonal(K_learned, 0)
        K_learned = np.clip(K_learned, 0, 1)  # keep non-negative

        # Track progress
        if s % 100 == 0:
            # Correlation between learned and target K
            mask = ~np.eye(N, dtype=bool)
            corr = np.corrcoef(K_learned[mask].flatten(), K_target[mask].flatten())[0, 1]
            R_val = float(np.abs(np.mean(np.exp(1j * theta))))
            correlation_history.append(float(corr))
            R_history.append(R_val)

    return {
        "K_learned": K_learned,
        "final_corr": correlation_history[-1] if correlation_history else 0,
        "corr_history": correlation_history,
        "R_history": R_history,
        "convergence_step": next(
            (i * 100 for i, c in enumerate(correlation_history) if c > 0.5), None
        ),
    }


print("Ready.")

In [ ]:
# Compare 3 learning modes
modes = ["standard", "fim_modulated", "fim_gated"]

print("=== FIM-MODULATED LEARNING ===")
print(f"{'Mode':>16} {'Final corr':>12} {'Conv step':>10} {'Final R':>8}")

learning_results = {}
for mode in modes:
    # Average over 5 seeds
    corrs = []
    steps = []
    Rs = []
    for seed in range(5):
        res = simulate_learning(N, K_target, omega, mode=mode, seed=seed)
        corrs.append(res["final_corr"])
        steps.append(res["convergence_step"])
        Rs.append(res["R_history"][-1] if res["R_history"] else 0)

    mean_corr = np.mean(corrs)
    valid_steps = [s for s in steps if s is not None]
    mean_step = np.mean(valid_steps) if valid_steps else float("inf")
    mean_R = np.mean(Rs)

    learning_results[mode] = {
        "corr": round(float(mean_corr), 4),
        "step": round(float(mean_step), 0) if valid_steps else "never",
        "R": round(float(mean_R), 4),
    }
    step_str = f"{mean_step:.0f}" if valid_steps else "never"
    print(f"{mode:>16} {mean_corr:12.4f} {step_str:>10} {mean_R:8.4f}")

print("\n=== VERDICT ===")
best = max(
    learning_results.items(), key=lambda x: x[1]["corr"] if isinstance(x[1]["corr"], float) else 0
)
print(f"Best learning mode: {best[0]} (corr={best[1]['corr']:.4f})")
if "fim" in best[0]:
    print("FIM-modulated learning OUTPERFORMS standard Hebbian.")
    print("Self-observation accelerates adaptation.")
else:
    print("Standard Hebbian is sufficient. FIM modulation not needed for learning.")

In [ ]:
# Save
output = {
    "experiment": "FIM-modulated learning — does self-observation accelerate adaptation?",
    "N": N,
    "results": learning_results,
}
out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/fim_modulated_learning_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2, default=str)
print(f"Results saved to {out_path}")